# Loss functions

> A set of custom loss functions

In [2]:
#| default_exp losses

In [3]:
#| hide
from nbdev.showdoc import *

# Loss Functions

In [4]:
#| export
from fastai.vision.all import *
from monai.losses import SSIMLoss


In [5]:
#| export
class CombinedLoss:
    "losses combined"
    def __init__(self, spatial_dims=2, alpha=0.33, beta=0.33):
        store_attr()
        self.SSIM_loss = SSIMLoss(spatial_dims=spatial_dims)
        self.MSE_loss =  nn.MSELoss()
        self.MAE_loss =  nn.L1Loss()
        
    def __call__(self, pred, targ):
        return (1 - self.alpha - self.beta) * self.SSIM_loss(pred, targ) + self.alpha * self.MSE_loss(pred, targ) + self.beta * self.MAE_loss(pred, targ)

In [8]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Asegúrate de que los inputs sean probabilidades
        inputs = torch.sigmoid(inputs)

        # Aplanar tensores
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calcula la intersección
        intersection = (inputs * targets).sum()

        # Calcula el coeficiente de Dice
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        # Calcula la pérdida de Dice
        loss = 1 - dice

        return loss

# Ejemplo de uso:
# inputs y targets deben ser tensores de las mismas dimensiones
# inputs es típicamente la salida de la red neuronal antes de aplicar una función de activación
inputs = torch.randn((1, 1, 256, 256))  # Ejemplo de entrada de red (predicciones)
targets = torch.randint(0, 2, (1, 1, 256, 256)).float()  # Ejemplo de ground truth

# Inicializa la función de pérdida
dice_loss = DiceLoss()

# Calcula la pérdida
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())

print(inputs)
print(targets)

Dice Loss: 0.49923235177993774
tensor([[[[ 0.3067, -0.3703, -0.4363,  ...,  0.6810, -0.0361, -1.6102],
          [-2.5675, -0.8505, -0.6124,  ..., -0.7215, -1.5011, -1.5442],
          [-1.9884, -0.4281,  0.3839,  ..., -0.3354, -1.1512,  0.3624],
          ...,
          [ 0.2546,  0.3077, -0.2814,  ...,  0.5136,  0.8059,  0.4465],
          [ 0.2610, -0.4132, -0.8067,  ...,  0.1354, -0.5225, -0.8665],
          [-0.4170, -0.9477,  0.6713,  ..., -0.2064, -0.1529, -0.3958]]]])
tensor([[[[1., 1., 1.,  ..., 0., 1., 1.],
          [1., 1., 0.,  ..., 1., 0., 1.],
          [0., 0., 0.,  ..., 0., 1., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [1., 0., 0.,  ..., 1., 0., 1.],
          [0., 1., 1.,  ..., 0., 1., 1.]]]])


# Metrics

In [5]:
#| export

def SSIM(x, y, spatial_dims=2):
    return 1 - SSIMLoss(spatial_dims)(x,y)

SSIMMetric = AvgMetric(SSIM)

In [6]:
#| hide
import nbdev; nbdev.nbdev_export()